In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

results = pd.read_csv("../package_metadata/results_all.csv")

# Image 1

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from math import ceil

all_explainers = ["expected_gradients", "shap_kernel", "sage_permutation", "shapiq_kernel"]
all_metrics = ["explanation_time", "mae", "top_k", "mmd", "compression_time"]
all_models = ['nn', 'xgboost']
methods = ["kt_gaussian", "kt_matern", "kt_inverse_multiquadric"]

sns.set_theme(style="whitegrid")
bright_colors = sns.color_palette("bright", n_colors=len(methods))

base_dir = "Image1"
os.makedirs(base_dir, exist_ok=True)

for model_name in all_models:
    model_dir = os.path.join(base_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)
    
    for explainer_total in all_explainers:
        for metric in all_metrics:

            mask = (
                (results['explainer_total'] == explainer_total) &
                (results['model_name'] == model_name) &
                (results['method_total'].isin(methods))
            )
            filtered_results = results[mask]
            
            if filtered_results.empty:
                continue

            regression_list = []
            classification_list = []
            
            unique_datasets = sorted(filtered_results['dataset_name'].unique())
            for dataset in unique_datasets:
                ds_data = filtered_results[filtered_results['dataset_name'] == dataset]
                if not ds_data.empty and ds_data[metric].notna().any():
                    task_type = ds_data['task'].iloc[0] if 'task' in ds_data.columns else 'unknown'
                    if task_type == 'regression':
                        regression_list.append(dataset)
                    else:
                        classification_list.append(dataset)

            available_datasets = regression_list + classification_list
            if not available_datasets:
                continue

            n_cols = 4
            n_rows = ceil(len(available_datasets) / n_cols)

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 8 * n_rows))
            axes = np.atleast_1d(axes).flatten()

            legend_elements = []
            legend_labels = []

            for idx, dataset_name in enumerate(available_datasets):
                ax = axes[idx]
                is_regression = dataset_name in regression_list
                
                ax.set_facecolor('#f2f8ff' if is_regression else '#fffafa')
                
                data_plot = filtered_results[filtered_results['dataset_name'] == dataset_name]
                grouped = data_plot.groupby(['size', 'method_total'])[metric].agg(['mean', 'std', 'count']).reset_index()
                grouped['std_error'] = grouped['std'] / np.sqrt(grouped['count'])

                for m_idx, method in enumerate(methods):
                    method_data = grouped[grouped['method_total'] == method]
                    if method_data.empty:
                        continue
                        
                    line = ax.errorbar(
                        x=method_data['size'], 
                        y=method_data['mean'], 
                        yerr=method_data['std_error'],
                        label=method, 
                        marker='o', 
                        capsize=13,
                        markeredgewidth=3, 
                        linewidth=4.5,      
                        markersize=12,      
                        alpha=0.9,
                        color=bright_colors[m_idx],
                        markeredgecolor=bright_colors[m_idx],
                        elinewidth=2.5 
                    )
                    
                    if idx == 0:
                        legend_elements.append(line)
                        legend_labels.append(method)
                
                ax.set_xscale('log')
                ax.set_title(f'{dataset_name}', fontsize=40, fontweight='bold', pad=30)
                ax.set_xlabel('Sample Size', fontsize=36, fontweight='bold')
                ax.set_ylabel(f'{metric.upper()}', fontsize=36, fontweight='bold')
                
                ax.tick_params(axis='both', which='major', labelsize=32) 
                ax.tick_params(axis='both', which='minor', labelsize=24)
                
                ax.grid(True, which="both", alpha=0.6, linestyle='--', color='gray')
                ax.xaxis.set_major_formatter(plt.ScalarFormatter())
                ax.ticklabel_format(style='sci', axis='y', scilimits=(-2,2))
                ax.yaxis.get_offset_text().set_fontsize(30)

            for i in range(len(available_datasets), len(axes)):
                axes[i].set_visible(False)

            if legend_elements:
                fig.legend(legend_elements, legend_labels, loc='lower center', ncol=len(methods), 
                           bbox_to_anchor=(0.5, -0.06), fontsize=36, title="METHODS", 
                           title_fontsize=40, frameon=True, shadow=True, borderpad=1.5)

            plt.tight_layout(rect=[0, 0, 1, 1])
            plt.subplots_adjust(hspace=0.8, wspace=0.45)

            file_name = f"{explainer_total}_{metric}.png"
            plt.savefig(os.path.join(model_dir, file_name), bbox_inches='tight', dpi=100)
            plt.close(fig) 

print("Process finished. All images are saved in Image1 folder.")

Process finished. All images are saved in Image1 folder.
